<a href="https://colab.research.google.com/github/sudenurcure/ICR-Identifying-Age-Related-Conditions/blob/sudenur/Ac%C4%B1badem_Doktorlar_WS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install aiohttp pandas beautifulsoup4

In [3]:
import asyncio
import aiohttp
from bs4 import BeautifulSoup
import pandas as pd
import nest_asyncio
from google.colab import files

nest_asyncio.apply()

In [5]:
base_url = "https://www.acibadem.com.tr"
main_url = f"{base_url}/doktorlar/"

doctor_location_dict = []
doctor_service_dict = []

MAX_CONCURRENT_REQUESTS = 10

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}

async def fetch(session, url, retries=3):
    for attempt in range(retries):
        try:
            async with session.get(url, headers=headers) as response:
                response.raise_for_status()
                return await response.text()
        except aiohttp.ClientError as e:
            print(f"Attempt {attempt + 1} failed for {url}: {e}")
            if attempt == retries - 1:
                print(f"Failed to retrieve {url} after {retries} attempts.")
                return None
        await asyncio.sleep(2)

async def fetch_doctor_details(session, doctor_name, doctor_url):
    page_content = await fetch(session, doctor_url)
    if page_content is None:
        return

    doctor_soup = BeautifulSoup(page_content, 'html.parser')

    location_tags = doctor_soup.find('p', class_='doctor locations')
    if location_tags:
        for location in location_tags.find_all('a'):
            doctor_location_dict.append({
                'Doctor Name': doctor_name,
                'Location': location.get_text(strip=True)
            })

    service_tags = doctor_soup.find('p', class_='doctor units main top')
    if service_tags:
        for service in service_tags.find_all('a'):
            doctor_service_dict.append({
                'Doctor Name': doctor_name,
                'Service': service.get_text(strip=True)
            })

async def main():
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    async with aiohttp.ClientSession() as session:
        main_page = await fetch(session, main_url)
        if main_page is None:
            print("Failed to retrieve the main page.")
            return

        soup = BeautifulSoup(main_page, 'html.parser')
        doctor_list = soup.find('ul', class_='doctorList')
        doctor_links = doctor_list.find_all('a', class_='appointmentLink', href=True)

        tasks = []
        for link in doctor_links:
            doctor_name = link['title']
            doctor_url = base_url + link['href']
            task = asyncio.create_task(fetch_doctor_with_semaphore(session, doctor_name, doctor_url, semaphore))
            tasks.append(task)

        await asyncio.gather(*tasks)

async def fetch_doctor_with_semaphore(session, doctor_name, doctor_url, semaphore):
    async with semaphore:
        await fetch_doctor_details(session, doctor_name, doctor_url)
        await asyncio.sleep(1)

# Run the main function
asyncio.run(main())

# Save data to Excel
table1 = pd.DataFrame(doctor_location_dict)
table2 = pd.DataFrame(doctor_service_dict)

with pd.ExcelWriter('doctors_info_optimized.xlsx') as writer:
    table1.to_excel(writer, sheet_name='Doctor Locations', index=False)
    table2.to_excel(writer, sheet_name='Doctor Services', index=False)

print("Data has been successfully collected and saved to 'doctors_info_optimized.xlsx'.")

# Download the Excel file
files.download('doctors_info_optimized.xlsx')


Data has been successfully collected and saved to 'doctors_info_optimized.xlsx'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>